# 05 — Gold Aggregations

**Week:** 7

Goal: Create dashboard-ready Gold metric tables.


In [0]:
%sql

CREATE OR REPLACE TABLE gold_dim_property
USING DELTA
AS
SELECT
    property_id,
    MAX(property_name) AS property_name
FROM silver_rooms_trusted
GROUP BY property_id;


In [0]:
%sql

SELECT
    COUNT(*) AS property_rows,
    COUNT(DISTINCT property_id) AS unique_properties
FROM gold_dim_property;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_room
USING DELTA
AS
SELECT
    room_id,
    property_id,
    room_type,
    capacity
FROM silver_rooms_trusted;

In [0]:
%sql
SELECT
    COUNT(*) AS room_rows,
    COUNT(DISTINCT room_id) AS unique_rooms,
    COUNT(DISTINCT property_id) AS properties
FROM gold_dim_room;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_booking_summary
USING DELTA
AS
SELECT
    property_id,
    booking_status,
    COUNT(*) AS booking_count,
    COUNT(DISTINCT guest_id) AS unique_guests,
    COUNT(DISTINCT requested_room_type) AS unique_rooms,
    SUM(booked_amount) AS total_booking_value,
    AVG(booked_amount) AS avg_booking_value
FROM silver_bookings_trusted
GROUP BY
    property_id,
    booking_status;

In [0]:
%sql
SELECT *
FROM gold_booking_summary
ORDER BY property_id, booking_status;

In [0]:
%sql
DESCRIBE silver_room_nights_trusted;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_room_night_summary
USING DELTA
AS
SELECT
    property_id,
    stay_date,
    COUNT(*) AS room_night_count,
    SUM(occupied_flag) AS occupied_room_nights,
    SUM(available_flag) AS available_room_nights,
    SUM(revenue_eligible_flag) AS revenue_eligible_room_nights,
    SUM(recognized_room_revenue) AS total_room_revenue,
    CASE
        WHEN SUM(available_flag) > 0
        THEN ROUND(
            SUM(occupied_flag) * 100.0 / SUM(available_flag),
            2
        )
        ELSE 0
    END AS occupancy_rate_pct
FROM silver_room_nights_trusted
GROUP BY
    property_id,
    stay_date;

In [0]:
%sql
SELECT *
FROM gold_room_night_summary
ORDER BY property_id, stay_date
LIMIT 20;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_property_kpis
USING DELTA
AS
SELECT
    property_id,
    COUNT(DISTINCT stay_date) AS total_days,
    SUM(room_night_count) AS total_room_nights,
    SUM(occupied_room_nights) AS occupied_room_nights,
    SUM(available_room_nights) AS available_room_nights,
    SUM(revenue_eligible_room_nights) AS revenue_eligible_room_nights,
    SUM(total_room_revenue) AS total_room_revenue,
    ROUND(AVG(occupancy_rate_pct), 2) AS avg_occupancy_rate_pct
FROM gold_room_night_summary
GROUP BY property_id;

In [0]:
%sql
SELECT *
FROM gold_property_kpis
ORDER BY property_id;

In [0]:
%sql
SELECT
    'gold_dim_property' AS table_name,
    COUNT(*) AS row_count
FROM gold_dim_property

UNION ALL

SELECT
    'gold_dim_room',
    COUNT(*)
FROM gold_dim_room

UNION ALL

SELECT
    'gold_booking_summary',
    COUNT(*)
FROM gold_booking_summary

UNION ALL

SELECT
    'gold_room_night_summary',
    COUNT(*)
FROM gold_room_night_summary

UNION ALL

SELECT
    'gold_property_kpis',
    COUNT(*)
FROM gold_property_kpis
ORDER BY table_name;

In [0]:
%sql
-- CELL 14: Gold Dimensions + Validation

CREATE OR REPLACE TABLE gold_dim_room_type
USING DELTA
AS
SELECT
    property_id,
    room_type,
    MAX(capacity) AS standard_capacity
FROM silver_rooms_trusted
GROUP BY property_id, room_type;

CREATE OR REPLACE TABLE gold_dim_guest_segment
USING DELTA
AS
SELECT DISTINCT
    UPPER(TRIM(COALESCE(market_segment, 'UNKNOWN'))) AS guest_segment
FROM silver_bookings_trusted;

-- Validation
SELECT
    'gold_dim_room_type' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT CONCAT(property_id, '|', room_type)) AS unique_key_count
FROM gold_dim_room_type

UNION ALL

SELECT
    'gold_dim_guest_segment' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT guest_segment) AS unique_key_count
FROM gold_dim_guest_segment;

In [0]:
%sql
-- CELL 15: Gold Facts + Validation

-- 1. Booking Fact: one row per trusted booking
CREATE OR REPLACE TABLE gold_fact_booking
USING DELTA
AS
SELECT
    booking_id,
    property_id,
    guest_id,
    requested_room_type,
    rate_plan_id,
    booking_date,
    arrival_date,
    departure_date,
    booking_status,
    market_segment,
    channel,
    adults,
    children,
    rooms_booked,
    nightly_rate,
    discount_amount,
    tax_amount,
    refund_amount,
    booked_amount,
    net_booking_value,

    DATEDIFF(departure_date, arrival_date) AS stay_nights,

    COALESCE(adults, 0) + COALESCE(children, 0) AS party_size,

    DATE_TRUNC('month', booking_date) AS booking_month,

    CASE
        WHEN DATEDIFF(departure_date, arrival_date) <= 2 THEN 'SHORT'
        WHEN DATEDIFF(departure_date, arrival_date) <= 5 THEN 'MEDIUM'
        ELSE 'LONG'
    END AS stay_length_band

FROM silver_bookings_trusted
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY booking_id
    ORDER BY booking_date DESC, arrival_date DESC
) = 1;


-- 2. Room-Night Fact: one row per trusted room-night
CREATE OR REPLACE TABLE gold_fact_room_night
USING DELTA
AS
SELECT
    room_night_id,
    booking_id,
    property_id,
    room_id,
    room_type,
    stay_date,
    rate_plan_id,
    occupied_flag,
    available_flag,
    revenue_eligible_flag,
    complimentary_flag,
    out_of_service_flag,
    recognized_room_revenue,

    CASE
        WHEN available_flag = 1
         AND out_of_service_flag = 0
        THEN 1
        ELSE 0
    END AS available_for_sale_flag

FROM silver_room_nights_trusted;


-- Validation
SELECT
    'gold_fact_booking' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT booking_id) AS unique_key_count
FROM gold_fact_booking

UNION ALL

SELECT
    'gold_fact_room_night' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT room_night_id) AS unique_key_count
FROM gold_fact_room_night;

In [0]:
%sql
-- CELL 16: Daily Gold Summaries

-- 1. Daily Property Summary
CREATE OR REPLACE TABLE gold_daily_property_summary
USING DELTA AS
SELECT
    property_id,
    stay_date,
    SUM(occupied_flag) AS occupied_room_nights,
    SUM(
        CASE
            WHEN available_flag = 1 AND out_of_service_flag = 0
            THEN 1 ELSE 0
        END
    ) AS available_room_nights,
    SUM(
        CASE
            WHEN revenue_eligible_flag = 1
            THEN recognized_room_revenue
            ELSE 0
        END
    ) AS recognized_room_revenue,
    CASE
        WHEN SUM(
            CASE
                WHEN available_flag = 1 AND out_of_service_flag = 0
                THEN 1 ELSE 0
            END
        ) > 0
        THEN ROUND(
            SUM(occupied_flag) * 100.0 /
            SUM(
                CASE
                    WHEN available_flag = 1 AND out_of_service_flag = 0
                    THEN 1 ELSE 0
                END
            ), 2
        )
        ELSE 0
    END AS occupancy_rate_pct
FROM gold_fact_room_night
GROUP BY property_id, stay_date;


-- 2. Daily Room-Type Summary
CREATE OR REPLACE TABLE gold_daily_room_type_summary
USING DELTA AS
SELECT
    property_id,
    room_type,
    stay_date,
    SUM(occupied_flag) AS occupied_room_nights,
    SUM(
        CASE
            WHEN available_flag = 1 AND out_of_service_flag = 0
            THEN 1 ELSE 0
        END
    ) AS available_room_nights,
    SUM(
        CASE
            WHEN revenue_eligible_flag = 1
            THEN recognized_room_revenue
            ELSE 0
        END
    ) AS recognized_room_revenue
FROM gold_fact_room_night
GROUP BY property_id, room_type, stay_date;


-- 3. Daily Channel / Segment Summary
CREATE OR REPLACE TABLE gold_daily_channel_segment_summary
USING DELTA AS
SELECT
    booking_date,
    channel,
    market_segment,
    COUNT(DISTINCT booking_id) AS total_bookings,
    SUM(booked_amount) AS total_booking_value,
    AVG(booked_amount) AS avg_booking_value
FROM gold_fact_booking
GROUP BY booking_date, channel, market_segment;


-- 4. Daily Rate-Plan Summary
CREATE OR REPLACE TABLE gold_daily_rate_plan_summary
USING DELTA AS
SELECT
    property_id,
    rate_plan_id,
    stay_date,
    SUM(occupied_flag) AS occupied_room_nights,
    SUM(
        CASE
            WHEN revenue_eligible_flag = 1
            THEN recognized_room_revenue
            ELSE 0
        END
    ) AS recognized_room_revenue
FROM gold_fact_room_night
GROUP BY property_id, rate_plan_id, stay_date;


-- Validation
SELECT 'gold_daily_property_summary' AS table_name, COUNT(*) AS row_count
FROM gold_daily_property_summary
UNION ALL
SELECT 'gold_daily_room_type_summary', COUNT(*)
FROM gold_daily_room_type_summary
UNION ALL
SELECT 'gold_daily_channel_segment_summary', COUNT(*)
FROM gold_daily_channel_segment_summary
UNION ALL
SELECT 'gold_daily_rate_plan_summary', COUNT(*)
FROM gold_daily_rate_plan_summary;

In [0]:
%sql
-- CELL 17: Final Gold KPI Model + Validation

CREATE OR REPLACE TABLE gold_property_kpis_final
USING DELTA
AS

WITH booking_kpis AS (
    SELECT
        property_id,

        COUNT(DISTINCT booking_id) AS total_bookings,

        COUNT(DISTINCT CASE
            WHEN UPPER(TRIM(booking_status)) = 'CANCELLED'
            THEN booking_id
        END) AS cancelled_bookings,

        COUNT(DISTINCT CASE
            WHEN UPPER(TRIM(booking_status)) = 'NO-SHOW'
            THEN booking_id
        END) AS no_show_bookings,

        COUNT(DISTINCT CASE
            WHEN UPPER(TRIM(booking_status)) NOT IN ('CANCELLED')
            THEN booking_id
        END) AS no_show_eligible_bookings,

        AVG(CASE
            WHEN UPPER(TRIM(booking_status))
                 NOT IN ('CANCELLED', 'NO-SHOW')
            THEN stay_nights
        END) AS average_length_of_stay

    FROM gold_fact_booking
    GROUP BY property_id
),

room_night_kpis AS (
    SELECT
        property_id,

        SUM(occupied_flag) AS occupied_room_nights,

        SUM(
            CASE
                WHEN available_flag = 1
                 AND out_of_service_flag = 0
                THEN 1
                ELSE 0
            END
        ) AS available_room_nights,

        SUM(
            CASE
                WHEN revenue_eligible_flag = 1
                THEN recognized_room_revenue
                ELSE 0
            END
        ) AS recognized_room_revenue

    FROM gold_fact_room_night
    GROUP BY property_id
)

SELECT
    p.property_id,

    COALESCE(b.total_bookings, 0) AS total_bookings,

    COALESCE(r.occupied_room_nights, 0) AS occupied_room_nights,

    COALESCE(r.available_room_nights, 0) AS available_room_nights,

    COALESCE(r.recognized_room_revenue, 0) AS recognized_room_revenue,

    CASE
        WHEN COALESCE(r.available_room_nights, 0) > 0
        THEN ROUND(
            r.occupied_room_nights * 100.0 /
            r.available_room_nights,
            2
        )
        ELSE 0
    END AS occupancy_rate_pct,

    CASE
        WHEN COALESCE(r.occupied_room_nights, 0) > 0
        THEN ROUND(
            r.recognized_room_revenue /
            r.occupied_room_nights,
            2
        )
        ELSE 0
    END AS adr,

    CASE
        WHEN COALESCE(r.available_room_nights, 0) > 0
        THEN ROUND(
            r.recognized_room_revenue /
            r.available_room_nights,
            2
        )
        ELSE 0
    END AS revpar,

    COALESCE(b.cancelled_bookings, 0) AS cancelled_bookings,

    CASE
        WHEN COALESCE(b.total_bookings, 0) > 0
        THEN ROUND(
            b.cancelled_bookings * 100.0 /
            b.total_bookings,
            2
        )
        ELSE 0
    END AS cancellation_rate_pct,

    COALESCE(b.no_show_bookings, 0) AS no_show_bookings,

    CASE
        WHEN COALESCE(b.no_show_eligible_bookings, 0) > 0
        THEN ROUND(
            b.no_show_bookings * 100.0 /
            b.no_show_eligible_bookings,
            2
        )
        ELSE 0
    END AS no_show_rate_pct,

    ROUND(COALESCE(b.average_length_of_stay, 0), 2)
        AS average_length_of_stay

FROM gold_dim_property p
LEFT JOIN booking_kpis b
    ON p.property_id = b.property_id
LEFT JOIN room_night_kpis r
    ON p.property_id = r.property_id;


-- Validation
SELECT
    COUNT(*) AS property_count,
    COUNT(DISTINCT property_id) AS unique_property_count,
    SUM(total_bookings) AS total_bookings,
    SUM(occupied_room_nights) AS occupied_room_nights,
    SUM(available_room_nights) AS available_room_nights,
    SUM(recognized_room_revenue) AS recognized_room_revenue
FROM gold_property_kpis_final;

In [0]:
%sql
-- CELL 18: Manual Spot-Check + Reconciliation

WITH anchor_date AS (
    SELECT
        MIN(property_id) AS property_id,
        MIN(stay_date) AS stay_date
    FROM gold_fact_room_night
),

room_source AS (
    SELECT
        f.property_id,
        f.stay_date,
        SUM(f.occupied_flag) AS occupied_room_nights,
        SUM(
            CASE
                WHEN f.available_flag = 1
                 AND f.out_of_service_flag = 0
                THEN 1 ELSE 0
            END
        ) AS available_room_nights,
        SUM(
            CASE
                WHEN f.revenue_eligible_flag = 1
                THEN f.recognized_room_revenue
                ELSE 0
            END
        ) AS recognized_room_revenue
    FROM gold_fact_room_night f
    CROSS JOIN anchor_date a
    WHERE f.property_id = a.property_id
      AND f.stay_date = a.stay_date
    GROUP BY f.property_id, f.stay_date
),

gold_daily AS (
    SELECT *
    FROM gold_daily_property_summary
    WHERE property_id = (SELECT property_id FROM anchor_date)
      AND stay_date = (SELECT stay_date FROM anchor_date)
),

anchor_booking AS (
    SELECT MIN(booking_id) AS booking_id
    FROM gold_fact_booking
),

booking_check AS (
    SELECT
        g.booking_id,
        g.property_id,
        g.booked_amount AS gold_booked_amount,
        s.booked_amount AS trusted_booked_amount,
        g.booking_status AS gold_status,
        s.booking_status AS trusted_status
    FROM gold_fact_booking g
    JOIN silver_bookings_trusted s
        ON g.booking_id = s.booking_id
    WHERE g.booking_id = (SELECT booking_id FROM anchor_booking)
)

-- Property/date reconciliation
SELECT
    'PROPERTY_DATE_CHECK' AS check_type,
    CONCAT(
        CAST((SELECT property_id FROM anchor_date) AS STRING),
        ' / ',
        CAST((SELECT stay_date FROM anchor_date) AS STRING)
    ) AS anchor_key,
    'occupied_room_nights' AS metric,
    CAST(r.occupied_room_nights AS STRING) AS trusted_fact_value,
    CAST(g.occupied_room_nights AS STRING) AS gold_summary_value,
    CASE
        WHEN r.occupied_room_nights = g.occupied_room_nights
        THEN 'PASS'
        ELSE 'FAIL'
    END AS result
FROM room_source r
JOIN gold_daily g
    ON r.property_id = g.property_id
   AND r.stay_date = g.stay_date

UNION ALL

SELECT
    'PROPERTY_DATE_CHECK',
    CONCAT(
        CAST((SELECT property_id FROM anchor_date) AS STRING),
        ' / ',
        CAST((SELECT stay_date FROM anchor_date) AS STRING)
    ),
    'available_room_nights',
    CAST(r.available_room_nights AS STRING),
    CAST(g.available_room_nights AS STRING),
    CASE
        WHEN r.available_room_nights = g.available_room_nights
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM room_source r
JOIN gold_daily g
    ON r.property_id = g.property_id
   AND r.stay_date = g.stay_date

UNION ALL

SELECT
    'PROPERTY_DATE_CHECK',
    CONCAT(
        CAST((SELECT property_id FROM anchor_date) AS STRING),
        ' / ',
        CAST((SELECT stay_date FROM anchor_date) AS STRING)
    ),
    'recognized_room_revenue',
    CAST(r.recognized_room_revenue AS STRING),
    CAST(g.recognized_room_revenue AS STRING),
    CASE
        WHEN r.recognized_room_revenue = g.recognized_room_revenue
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM room_source r
JOIN gold_daily g
    ON r.property_id = g.property_id
   AND r.stay_date = g.stay_date

UNION ALL

-- Anchor booking reconciliation
SELECT
    'ANCHOR_BOOKING_CHECK',
    booking_id,
    'booked_amount',
    CAST(trusted_booked_amount AS STRING),
    CAST(gold_booked_amount AS STRING),
    CASE
        WHEN trusted_booked_amount = gold_booked_amount
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM booking_check

UNION ALL

SELECT
    'ANCHOR_BOOKING_CHECK',
    booking_id,
    'booking_status',
    trusted_status,
    gold_status,
    CASE
        WHEN trusted_status = gold_status
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM booking_check;

## Update

Document metric formulas in `docs/gold_metrics_definition.md`.
